# Imports

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import weibull_min, norm
from sklearn.cluster import KMeans

# for replication
np.random.seed(42)

In [118]:
df = pd.read_csv("/Users/awaischoudhry/Repos/AMO_group_project/produkt_ff_stunde_19570701_20241231_02667.txt", sep=";") #src: https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/hourly/wind/historical/

In [119]:
df

,STATIONS_ID,MESS_DATUM,QN_3,F,D,eor
0,2667,1957090100,5,1.3,-999,eor
1,2667,1957090101,5,0.9,-999,eor
2,2667,1957090102,5,1.3,-999,eor
3,2667,1957090103,5,0.9,-999,eor
4,2667,1957090104,5,1.1,-999,eor
...,...,...,...,...,...,...
588901,2667,2024123119,10,3.7,170,eor
588902,2667,2024123120,10,5.4,180,eor
588903,2667,2024123121,10,6.2,180,eor
588904,2667,2024123122,10,5.0,160,eor


# Data Wrangling

In [120]:
df['datetime'] = pd.to_datetime(df['MESS_DATUM'], format='%Y%m%d%H')

In [121]:
df["hour"] = df["datetime"].dt.hour
df["month"] = df["datetime"].dt.month

In [122]:
# select some period for wind
winter = df["month"].isin([12,1,2])
df_sel = df[winter & df["hour"].isin([10,11,12])]

In [123]:
wind = df_sel["   F"].values

# Scenario Generation

### Distribution fitting, MC and Copula

In [ ]:
def create_blocks(data, block_length=3):
    """ get snippets of wind series of length = block_length """
    return np.array([data[i:i+block_length] 
                     for i in range(len(data)-block_length+1)])

# choose 3 for t=3
blocks = create_blocks(wind, 3)

# calculate the cdf prob value for each single element, independently
u = weibull_min.cdf(blocks, shape, loc, scale)

In [124]:
# MLE parameter estimation for a weibull dist to fit wind behavior
shape, loc, scale = weibull_min.fit(wind, floc=0)

In [126]:
# the observed wind series snippets
blocks

array([[4.6, 5.3, 5.9],
       [5.3, 5.9, 0. ],
       [5.9, 0. , 0.1],
       ...,
       [4.6, 4.8, 2.8],
       [4.8, 2.8, 2.4],
       [2.8, 2.4, 2.7]])

In [127]:
# their cdf probabilities
u

array([[6.45335821e-01, 7.47037853e-01, 8.17660513e-01],
       [7.47037853e-01, 8.17660513e-01, 0.00000000e+00],
       [8.17660513e-01, 0.00000000e+00, 5.04949792e-04],
       ...,
       [6.45335821e-01, 6.76416436e-01, 3.19950483e-01],
       [6.76416436e-01, 3.19950483e-01, 2.46961609e-01],
       [3.19950483e-01, 2.46961609e-01, 3.01376591e-01]])

In [128]:
#implementation of a Gaussian Copula. Its purpose is to generate new, synthetic wind speeds that maintain the same correlation structure as the original data.

n_mc = 1000

# generate correlated normals
eps = 1e-10
u = np.clip(u, eps, 1 - eps)
# Converts probabilities to percentiles of standard Gaussian distribution.
z = norm.ppf(u)

#z.T is (3,1000), where each of the 3 lines is for one t. thus, calculates the Pearson correlation matrix between each t, of Gaussian-transformed data. The result is a 3×3 matrix.
corr = np.corrcoef(z.T)
#for numerical stability purposes
corr = (corr + corr.T) / 2
corr += 1e-6 * np.eye(3)

#generate outcomes of a multivariate normal, reflecting the covariance structure of wind speed values. 
z_samples = np.random.multivariate_normal(
    mean=np.zeros(3),
    cov=corr,
    size=n_mc,
)

# transform back to wind speed values
u_samples = norm.cdf(z_samples)
wind_samples = weibull_min.ppf(u_samples, shape, loc, scale)

In [129]:
wind_samples.shape = (1000, 3)

In [130]:
# the simulated wind speed samples from the fitted weibull dist
wind_samples

array([[3.02656672, 2.37222577, 3.31515393],
       [1.259368  , 1.39384453, 1.58048222],
       [1.8479221 , 1.43924763, 0.85289214],
       ...,
       [2.72457949, 2.7724266 , 2.58576377],
       [5.84554761, 5.61034703, 5.64545817],
       [6.5847807 , 6.68502182, 5.70587865]])

In [131]:
#cluster the samples to 2, because we want to intra day scenarios

kmeans = KMeans(n_clusters=2, random_state=42)
labels = kmeans.fit_predict(wind_samples)

intra_scenarios = kmeans.cluster_centers_


/opt/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


In [132]:
# the 2 intra day scenarios for each of the 3 time periods
intra_scenarios

array([[2.81216564, 2.65240452, 2.83690025],
       [5.62449779, 5.66976912, 5.56608392]])

In [133]:
# probabilities of the scenarios
p_intra = np.array([
    np.mean(labels == i) for i in range(2)
])

p_intra

array([0.604, 0.396])

In [134]:
# wind speed day ahead prediction as expectation of intra scenarios

w_DA = np.sum(
    intra_scenarios * p_intra[:, None],
    axis=0
)

In [135]:
# wind speed day ahead forecast for each t
w_DA

array([3.92584917, 3.8472809 , 3.91765698])

### low, medium, high real time scenario

In [93]:
# calibrate the desired deviations of real time to intra
delta_low  = -0.15   # -15%
delta_mid  =  0.02   # +2%
delta_high =  0.15   # +15%

deltas = [delta_low, delta_mid, delta_high]

In [139]:
n_real = 3
scenario_tree = {}

# for each t and intra day scenario, generate the three real time scenarios
for omega in range(2):
    
    base_path = intra_scenarios[omega]
    rt_paths = []
    
    for d in deltas:
        noise = np.random.normal(0, 0.05, 3)
        path = base_path * (1 + d + noise)
        path = np.maximum(path, 0)   # physical bound, no negative values
        rt_paths.append(path)
    
    scenario_tree[omega] = np.array(rt_paths)


In [140]:
scenario_tree # 0: high intra day scenario , 1: low intra day scenario

{0: array([[2.32690412, 2.32771583, 2.58161635],
        [2.80328463, 2.65088912, 3.05731332],
        [2.97108932, 2.99873949, 3.28944596]]),
 1: array([[4.90715461, 4.67477944, 4.74075641],
        [5.03720816, 5.59655868, 5.80361898],
        [6.19192693, 6.53695583, 6.525363  ]])}

# Wind Speed to Power

In [141]:
# wind measured at 10m. Need to adjust to higher height
def adjust_to_hub_height(v, h_ref=10, h_hub=100, alpha=0.14):
    return v * (h_hub / h_ref)**alpha

In [142]:
# wind speed to power
def power_curve(v, v_ci=3, v_r=12, v_co=25, P_r=3):
    if v < v_ci or v >= v_co:
        return 0.0
    elif v <= v_r:
        return P_r * ((v - v_ci) / (v_r - v_ci))**3
    else:
        return P_r

In [143]:
vectorized_power = np.vectorize(power_curve)

In [144]:
n_turbines = 200

#### real time scenarios

In [145]:
scenario_tree_hub = {}

for omega in scenario_tree:
    scenario_tree_hub[omega] = adjust_to_hub_height(scenario_tree[omega])

In [146]:
scenario_tree_hub # 0: high id scenario , 1: low id scenario

{0: array([[3.21202184, 3.21314231, 3.56362259],
        [3.86960999, 3.65924563, 4.2202672 ],
        [4.10124495, 4.1394128 , 4.54069945]]),
 1: array([[6.77375901, 6.45299198, 6.54406556],
        [6.95328289, 7.72540153, 8.01122432],
        [8.5472385 , 9.02351097, 9.00750841]])}

In [147]:
scenario_tree_hub_power = scenario_tree_hub.copy()

In [148]:
for omega in scenario_tree:
    scenario_tree_hub_power[omega] = vectorized_power(scenario_tree_hub_power[omega])

In [149]:
# use a wind farm, as single turbine too weak
for omega in scenario_tree_hub_power:
    scenario_tree_hub_power[omega] *= n_turbines

#### intra scenarios

In [150]:
intra_scenarios_hub = adjust_to_hub_height(intra_scenarios)

In [151]:
intra_scenarios_hub

array([[3.8818692 , 3.66133747, 3.91601246],
       [7.76396825, 7.82646008, 7.68333466]])

In [152]:
intra_scenarios_hub_power = intra_scenarios_hub.copy()

In [153]:
intra_scenarios_hub_power = vectorized_power(intra_scenarios_hub)

In [154]:
intra_scenarios_hub_power*= n_turbines

In [155]:
intra_scenarios_hub_power

array([[ 0.564464  ,  0.23806366,  0.63259808],
       [88.98776466, 92.53581966, 84.54526716]])

### overview

In [156]:
p_DA = np.sum(
    intra_scenarios_hub_power * p_intra[:, None],
    axis=0
)

In [157]:
p_DA

array([35.58009106, 36.78797504, 33.86201503])

In [158]:
scenario_tree_hub_power

{0: array([[0.0078445 , 0.00796953, 0.14736315],
        [0.54124923, 0.23581179, 1.49550728],
        [1.09919693, 1.2174945 , 3.01007629]]),
 1: array([[ 44.23298113,  33.8852294 ,  36.63776407],
        [ 50.85075432,  86.84401128, 103.5750741 ],
        [140.49285514, 179.87584155, 178.44602776]])}